# 0. Extraction des donnees GDELT
Ce notebook interroge directement Google BigQuery pour extraire les donnees brutes.
**Points d'attention integres :**
- Filtre explicite pour **exclure "Benin City"** (Nigeria) qui polluerait nos donnees.
- Extraction des variables avancees V2 (V2Themes, V2Tone, QuadClass, etc.)
- Integration des pays de l'AES et frontaliers pour la comparaison.

In [1]:
from google.colab import auth
auth.authenticate_user()
print('Authentifié avec succès')

Authentifié avec succès


In [2]:
from google.cloud import bigquery
import pandas as pd
import os

project_id = 'gen-lang-client-0835094546'  # ✅ ton vrai ID

try:
    client = bigquery.Client(project=project_id)
    
    # Test immédiat
    test = client.query("SELECT 1 AS ping").to_dataframe()
    print(f"✅ Connecté à BigQuery — projet : {project_id}")

except Exception as e:
    print(f"❌ Erreur : {e}")


✅ Connecté à BigQuery — projet : gen-lang-client-0835094546


In [3]:
# Dossier persistant via Google Drive (optionnel)
from google.colab import drive
drive.mount('/content/drive')
data_dir = '/content/drive/MyDrive/data/raw'
os.makedirs(data_dir, exist_ok=True)
print(f"Dossier prêt : {data_dir}")

Mounted at /content/drive
Dossier prêt : /content/drive/MyDrive/data/raw


## 1. Table `events` : Les faits bruts enrichis
Ici nous filtrons avec les codes pays exacts, et pour être sûr on exclut textuellement "Benin City".
Et nous gardons des variables clés supplémentaires comme `QuadClass` (Type fondamental de conflit/coopération) et `NumSources` (L'importance médiatique).

In [4]:
query_events = """
SELECT
  SQLDATE, DATEADDED, QuadClass, IsRootEvent,
  Actor1Name, Actor1CountryCode, Actor1Type1Code,
  Actor2Name, Actor2CountryCode, Actor2Type1Code,
  EventCode, EventBaseCode, EventRootCode,
  GoldsteinScale, NumMentions, NumSources,
  NumArticles, AvgTone,
  ActionGeo_Type, ActionGeo_FullName, ActionGeo_CountryCode,
  ActionGeo_Lat, ActionGeo_Long, ActionGeo_FeatureID,
  SOURCEURL

FROM `gdelt-bq.gdeltv2.events_partitioned`
WHERE
  _PARTITIONDATE BETWEEN '2025-01-01' AND '2025-12-31'
  AND (
    Actor1CountryCode = 'BEN'
    OR Actor2CountryCode = 'BEN'
    OR ActionGeo_CountryCode = 'BN'
  )
ORDER BY SQLDATE
"""

import os
os.makedirs('/content/data/raw', exist_ok=True)

print("Téléchargement de la table 'events' en cours...")
df_events = client.query(query_events).to_dataframe()

output_path = '/content/data/raw/benin_events.csv'
df_events.to_csv(output_path, index=False)
print(f"✅ {len(df_events)} événements enregistrés → '{output_path}'")

Téléchargement de la table 'events' en cours...
✅ 31504 événements enregistrés → '/content/data/raw/benin_events.csv'


## 2. Table `gkg` : Thèmes et Extractions
La table `gkg` encode pour chaque article collecté ses métadonnées sémantiques : thèmes, personnes, organisations, localisation et tonalité. L'objectif est d'en extraire les entrées relatives au Bénin sur 2025.

**Variables initialement prévues** : `DATE`, `DocumentIdentifier`, `SourceCollectionIdentifier`, `Themes`, `Locations`, `Persons`, `Organizations`, `Tone`, `V2Themes`, `V2Locations`, `V2Persons`, `V2Organizations`, `V2Tone`, `GCAM`, `AllNames`, `Amounts`.

La table `gkg_partitioned` reste volumineuse même avec `_PARTITIONDATE`. Le coupable identifié est la colonne **`GCAM`** : elle encode des centaines de scores linguistiques par article (plusieurs Ko par cellule → plusieurs TB sur l'année).

**Arbitrage retenu** :

| Supprimé | Raison | Remplacé par |
|---|---|---|
| `GCAM` | Scores linguistiques fins — trop dense | `V2Tone` (Tone, Positive, Negative, Polarity, ActivityDensity, SelfDensity) |
| `AllNames` | Redondant | `V2Persons` + `V2Organizations` |
| `Amounts` | Chiffres bruts sans contexte | — |
| `Themes`, `Locations`, `Persons`, `Organizations`, `Tone` (V1) | Remplacés par les versions V2 plus précises | Colonnes V2 correspondantes |

**Colonnes finales retenues** : `DATE`, `DocumentIdentifier`, `SourceCollectionIdentifier`, `V2Themes`, `V2Locations`, `V2Persons`, `V2Organizations`, `V2Tone`.dans les articles de presse, sans omettre le moindre lien ! 

In [5]:
query_gkg = """
SELECT
  DATE,
  DocumentIdentifier,
  SourceCollectionIdentifier,
  V2Themes,
  V2Locations,
  V2Persons,
  V2Organizations,
  V2Tone

FROM `gdelt-bq.gdeltv2.gkg_partitioned`
WHERE
  _PARTITIONDATE BETWEEN '2025-01-01' AND '2025-12-31'
  AND V2Locations LIKE '%#BN#%'
  AND DATE >= 20250101000000
  AND DATE <= 20251231235959
"""

print("⚠️  Table GKG — requête en cours...")
df_gkg = client.query(query_gkg).to_dataframe()

print(f"Lignes récupérées : {len(df_gkg)}")
print(f"Taille en mémoire : {df_gkg.memory_usage(deep=True).sum() / 1024**2:.1f} MB")

output_path = '/content/data/raw/benin_gkg.csv'
df_gkg.to_csv(output_path, index=False)
print(f"✅ {len(df_gkg)} entrées enregistrées → '{output_path}'")

⚠️  Table GKG — requête en cours...
Lignes récupérées : 47921
Taille en mémoire : 148.4 MB
✅ 47921 entrées enregistrées → '/content/data/raw/benin_gkg.csv'


## 3. Table de `baseline` (La Comparaison AES +  Pays frontaliers)
Pour répondre à l'Insight 5 (Le Bénin vs Ses voisins), nous tirerons les indicateurs pour le Nigéria, le Niger, le Burkina Faso, le Mali, le Sénégal, le Togo et le Ghana :

In [8]:
query_baseline = """
SELECT
  ActionGeo_CountryCode AS fips_pays,
  EXTRACT(MONTH FROM PARSE_DATE('%Y%m%d', CAST(SQLDATE AS STRING))) AS mois,
  COUNT(*) AS nb_evenements,
  AVG(AvgTone) AS ton_moyen,
  AVG(GoldsteinScale) AS goldstein_moyen
FROM `gdelt-bq.gdeltv2.events_partitioned`
WHERE
  _PARTITIONDATE BETWEEN '2025-01-01' AND '2025-12-31'
  AND ActionGeo_CountryCode IN (
    'BN',  -- Bénin
    'NG',  -- Niger
    'ML',  -- Mali
    'UV',  -- Burkina Faso
    'NI',  -- Nigéria
    'TO',  -- Togo
    'GH',  -- Ghana
    'SG'   -- Sénégal
  )
GROUP BY fips_pays, mois
ORDER BY mois, fips_pays
"""
print("Téléchargement baseline régionale en cours...")
df_baseline = client.query(query_baseline).to_dataframe()
print(f"Lignes récupérées : {len(df_baseline)}")
output_path = '/content/data/raw/comparatif_regional.csv'
df_baseline.to_csv(output_path, index=False)
print(f"✅ {len(df_baseline)} mois d'indicateurs enregistrés → '{output_path}'")

Téléchargement baseline régionale en cours...
Lignes récupérées : 96
✅ 96 mois d'indicateurs enregistrés → '/content/data/raw/comparatif_regional.csv'


In [9]:
query_baseline = """
SELECT
  ActionGeo_CountryCode AS fips_pays,
  EXTRACT(MONTH FROM PARSE_DATE('%Y%m%d', CAST(SQLDATE AS STRING))) AS mois,
  
  COUNT(*) AS nb_evenements,
  AVG(AvgTone) AS ton_moyen,
  AVG(GoldsteinScale) AS goldstein_moyen,
  
  -- Volatilité du ton (écart-type) → détecte les mois instables
  STDDEV(AvgTone) AS ton_volatilite,
  
  -- Part des événements conflictuels (QuadClass 3 = conflit verbal, 4 = conflit matériel)
  COUNTIF(QuadClass IN (3, 4)) AS nb_conflits,
  ROUND(COUNTIF(QuadClass IN (3, 4)) / COUNT(*) * 100, 2) AS pct_conflits,
  
  -- Part des événements coopératifs (QuadClass 1 = déclaration, 2 = coopération)
  COUNTIF(QuadClass IN (1, 2)) AS nb_cooperation,
  ROUND(COUNTIF(QuadClass IN (1, 2)) / COUNT(*) * 100, 2) AS pct_cooperation

FROM `gdelt-bq.gdeltv2.events_partitioned`
WHERE
  _PARTITIONDATE BETWEEN '2025-01-01' AND '2025-12-31'
  AND ActionGeo_CountryCode IN (
    'BN', 'NG', 'ML', 'UV', 'NI', 'TO', 'GH', 'SG'
  )
GROUP BY fips_pays, mois
ORDER BY mois, fips_pays
"""

print("Téléchargement baseline régionale en cours...")
df_baseline = client.query(query_baseline).to_dataframe()
print(f"Lignes récupérées : {len(df_baseline)}")

output_path = '/content/data/raw/comparatif_regional_1.csv'
df_baseline.to_csv(output_path, index=False)
print(f"✅ {len(df_baseline)} mois d'indicateurs enregistrés → '{output_path}'")

Téléchargement baseline régionale en cours...
Lignes récupérées : 96
✅ 96 mois d'indicateurs enregistrés → '/content/data/raw/comparatif_regional_1.csv'


In [10]:
import pandas as pd

df_events   = pd.read_csv('/content/data/raw/benin_events.csv')
df_gkg      = pd.read_csv('/content/data/raw/benin_gkg.csv')
df_baseline = pd.read_csv('/content/data/raw/comparatif_regional.csv')
df_baseline_augmented = pd.read_csv('/content/data/raw/comparatif_regional_1.csv')


print("── EVENTS ──")
display(df_events.head())
print(f"Shape : {df_events.shape}\n")

print("── GKG ──")
display(df_gkg.head())
print(f"Shape : {df_gkg.shape}\n")

print("── BASELINE RÉGIONALE ──")
display(df_baseline.head())
print(f"Shape : {df_baseline.shape}")

print("── BASELINE AUGMENTÉE ──")
display(df_baseline_augmented.head())
print(f"Shape : {df_baseline_augmented.shape}") 

── EVENTS ──


,SQLDATE,DATEADDED,QuadClass,IsRootEvent,Actor1Name,Actor1CountryCode,Actor1Type1Code,Actor2Name,Actor2CountryCode,Actor2Type1Code,...,NumSources,NumArticles,AvgTone,ActionGeo_Type,ActionGeo_FullName,ActionGeo_CountryCode,ActionGeo_Lat,ActionGeo_Long,ActionGeo_FeatureID,SOURCEURL
0,20250101,20250101194500,1,1,NaN,NaN,NaN,BENIN,BEN,NaN,...,1,5,-8.482871,1,Benin,BN,9.5,2.25,BN,https://punchng.com/benin-republic-summons-nig...
1,20250101,20250101194500,3,1,NaN,NaN,NaN,BENIN,BEN,NaN,...,1,5,-8.482871,1,Benin,BN,9.5,2.25,BN,https://punchng.com/benin-republic-summons-nig...
2,20250101,20250101194500,1,0,BENIN,BEN,GOV,NaN,NaN,NaN,...,1,5,-8.482871,1,Benin,BN,9.5,2.25,BN,https://punchng.com/benin-republic-summons-nig...
3,20250101,20250101194500,1,1,DIPLOMAT,NaN,GOV,NaN,NaN,NaN,...,1,3,-7.843137,1,Benin,BN,9.5,2.25,BN,https://www.thecable.ng/benin-republic-summons...
4,20250101,20250101194500,1,1,BENIN,BEN,GOV,NaN,NaN,NaN,...,1,3,-8.482871,1,Niger,NG,16.0,8.00,NG,https://punchng.com/benin-republic-summons-nig...


Shape : (31504, 25)

── GKG ──


,DATE,DocumentIdentifier,SourceCollectionIdentifier,V2Themes,V2Locations,V2Persons,V2Organizations,V2Tone
0,20250728234500,https://www.wkyc.com/article/news/nation-world...,1,"SOC_SLAVERY,65;SOC_SLAVERY,578;SOC_SLAVERY,117...","4#Dakar, Dakar, Senegal#SG#SG01##14.7367#-17.6...","Yvon Detchenou,223;Atlantic Ocean,3320",NaN,"-1.32743362831858,2.8023598820059,4.1297935103..."
1,20250728234500,https://www.wnep.com/article/news/nation-world...,1,"LEGISLATION,37;LEGISLATION,118;LEGISLATION,496...","4#Bight Of Benin, Benin (General), Benin#BN#BN...","Atlantic Ocean,3320;Yvon Detchenou,223",NaN,"-1.32743362831858,2.8023598820059,4.1297935103..."
2,20250728234500,https://www.cbs19.tv/article/news/nation-world...,1,"TAX_FNCACT_CHILDREN,932;TAX_ETHNICITY_AMERICAN...","4#Bight Of Benin, Benin (General), Benin#BN#BN...","Atlantic Ocean,3320;Yvon Detchenou,223",NaN,"-1.32743362831858,2.8023598820059,4.1297935103..."
3,20250728234500,https://www.13wmaz.com/article/news/nation-wor...,1,"TAX_FNCACT_AUTHORITIES,1551;CRISISLEX_CRISISLE...","4#Bight Of Benin, Benin (General), Benin#BN#BN...","Yvon Detchenou,223;Atlantic Ocean,3320",NaN,"-1.32743362831858,2.8023598820059,4.1297935103..."
4,20250111234500,https://politicsnigeria.com/just-in-tcn-breaks...,1,"UNGP_FORESTS_RIVERS_OCEANS,712;AFFECT,647;AFFE...","4#Osogbo, Osun, Nigeria#NI#NI42#23012#7.76667#...",NaN,"Transmission Company Of Nigeria,35","-3.63636363636364,1.36363636363636,5,6.3636363..."


Shape : (47921, 8)

── BASELINE RÉGIONALE ──


,fips_pays,mois,nb_evenements,ton_moyen,goldstein_moyen
0,BN,1,2163,-1.635381,0.272631
1,GH,1,25962,-0.110928,1.274709
2,ML,1,4388,-1.453004,0.917229
3,NG,1,4356,-2.828742,0.602089
4,NI,1,107553,-1.266326,0.520319


Shape : (96, 5)
── BASELINE AUGMENTÉE ──


,fips_pays,mois,nb_evenements,ton_moyen,goldstein_moyen,ton_volatilite,nb_conflits,pct_conflits,nb_cooperation,pct_cooperation
0,BN,1,2163,-1.635381,0.272631,4.707641,614,28.39,1549,71.61
1,GH,1,25962,-0.110928,1.274709,4.711655,5064,19.51,20898,80.49
2,ML,1,4388,-1.453004,0.917229,3.904924,1150,26.21,3238,73.79
3,NG,1,4356,-2.828742,0.602089,4.316254,1179,27.07,3177,72.93
4,NI,1,107553,-1.266326,0.520319,4.776354,27892,25.93,79661,74.07


Shape : (96, 10)
